# Transparency Analysis Notes: 900 V New Run Comparison

This notebook keeps the original `Transparency.ipynb` dataset series and adds the newer 900 V data folders:

- `With Window 6_3 900V`
- `Without Window 6_5 900V`
- `Dark 06_07 900V`

The main metric remains normalized total accepted charge: `sum(accepted charges) / number of waveforms`. This includes both accepted-pulse size and accepted-pulse frequency, so it is the primary comparison quantity for transparency checks.


# Transparency: Original Series Plus 900 V New Runs


In [ ]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

from SRC.KingCRAB.pmt import find_voltage_folder
from SRC.KingCRAB.transparency import process_transparency_folder, get_result, TransparencyConfig, configure as configure_transparency, add_dataset_spec, result_value_and_error, plot_focused_group
from SRC.KingCRAB.pmt import (processing_files, normalize_waveform_times, subtract_baseline, lowpass_filter_waveform, filter_waveforms, extract_observables, integrate_pulse_region, waveform_duration, estimate_charge_peak, average_waveform as average_filtered_waveform)

# -----------------------------
# Imports
# -----------------------------
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

# Data Locations and Comparison Plan

The original voltage series is processed exactly as in `Transparency.ipynb`: `Dark`, `Without Window`, and `With Window` at 600, 700, 800, and 900 V.

Three additional 900 V datasets are appended from the external drive. The notebook prints the resolved folder and number of `C1C*.txt` files before processing, so path problems show up before the expensive analysis starts.


In [ ]:
# -----------------------------
# Data locations and settings - update only if the external drive path changes
# -----------------------------
untitled_base_folder = "/Volumes/Untitled"

dark_base_folder = os.path.join(untitled_base_folder, "Dark")
window_base_folder = os.path.join(untitled_base_folder, "With Window")
without_window_base_folder = os.path.join(untitled_base_folder, "Without Window")

new_window_900_folder = os.path.join(untitled_base_folder, "With Window 6_3 900V")
new_without_window_900_folder = os.path.join(untitled_base_folder, "Without Window 6_5 900V")
new_dark_900_folder = os.path.join(untitled_base_folder, "Dark 06_07 900V")

# Original Transparency.ipynb voltage series.
comparison_voltage_values = np.array([600, 700, 800, 900])
dark_voltage_values = comparison_voltage_values.copy()
voltage_values = comparison_voltage_values.copy()

original_test_labels = ["Without Window", "With Window"]
comparison_specs = [
    {"voltage": voltage, "reference_label": "Dark", "test_label": test_label}
    for voltage in comparison_voltage_values
    for test_label in original_test_labels
]

# New 900 V run-to-run comparisons. These drive the comparison table,
# accepted-charge overlays, accepted-waveform overlays, and ratio summary.
new_900_comparison_specs = [
    {"voltage": 900, "reference_label": "With Window", "test_label": "With Window 6_3"},
    {"voltage": 900, "reference_label": "Without Window", "test_label": "Without Window 6_5"},
    {"voltage": 900, "reference_label": "Dark", "test_label": "Dark 06_07"},
]
comparison_specs.extend(new_900_comparison_specs)

# Oscilloscope and analysis settings.
R_TERMINATION = 50.0
T_PRE_SIGNAL = 20e-9
PULSE_START_TIME = 20e-9
THRESHOLD_SIGMA = 5.0
APPLY_FILTER = True
F_CUTOFF = 1e9
QC_PERCENTILE = 5
CHARGE_HIST_BINS = 1000




configure_transparency(**globals())


dataset_specs = []
seen_datasets = set()

# Original dark series.
for voltage in dark_voltage_values:
    add_dataset_spec(
        dataset_specs,
        seen_datasets,
        "Dark",
        voltage,
        find_voltage_folder(dark_base_folder, voltage),
    )

# Original with/without-window series.
for voltage in comparison_voltage_values:
    add_dataset_spec(
        dataset_specs,
        seen_datasets,
        "Without Window",
        voltage,
        find_voltage_folder(without_window_base_folder, voltage),
    )
    add_dataset_spec(
        dataset_specs,
        seen_datasets,
        "With Window",
        voltage,
        find_voltage_folder(window_base_folder, voltage),
    )

# Additional 900 V datasets from the newer runs.
add_dataset_spec(dataset_specs, seen_datasets, "With Window 6_3", 900, new_window_900_folder, file_pattern="C1C*.txt")
add_dataset_spec(dataset_specs, seen_datasets, "Without Window 6_5", 900, new_without_window_900_folder, file_pattern="C3C*.txt")
add_dataset_spec(dataset_specs, seen_datasets, "Dark 06_07", 900, new_dark_900_folder, file_pattern="C3C*.txt")

for spec in dataset_specs:
    pattern = spec.get("file_pattern", "C1C*.txt")
    n_files = len(glob.glob(os.path.join(spec["folder"], pattern)))
    print(f"{spec['label']:20s} {spec['voltage']} V -> {spec['folder']} ({pattern}, {n_files} files)")

## Shared Waveform Processing

These functions are copied from the PMT processing workflow. Each waveform is split from the segmented text file, time-normalized, baseline-subtracted using the pre-pulse region, optionally filtered, and converted into pulse height and charge observables.


In [ ]:
# -----------------------------
# Waveform utilities copied from PMT_PROCESSING / Gain_Curve
# -----------------------------

## Dataset Metrics

`process_transparency_folder` is the main analysis function. It applies the height and charge cuts, stores all accepted charges, estimates the charge peak, computes average waveforms, and calculates normalized total accepted charge per waveform.

Important distinction: `Charge per waveform [C/wf]` is `total accepted charge / total waveform count`. `Mean accepted pulse charge [C/pulse]` is kept separately as a diagnostic for the average size of pulses that passed the cuts.


In [ ]:
# -----------------------------
# Total charge peak and accumulated-charge helpers
# -----------------------------

## Process All Datasets

This cell runs the analysis for every condition and voltage. The printed file, waveform, accepted-pulse, total-charge, and peak-charge values are the first sanity checks.


In [ ]:
# -----------------------------
# Process comparison datasets
# -----------------------------
transparency_results = []

for spec in dataset_specs:
    pattern = spec.get("file_pattern", "C1C*.txt")
    print(f"Processing {spec['label']} {spec['voltage']} V: {spec['folder']} ({pattern})")
    result = process_transparency_folder(spec["folder"], spec["label"], spec["voltage"], file_pattern=pattern)
    transparency_results.append(result)
    print(f"  status    : {result['Status']}")
    print(f"  files     : {result['Number of files']}")
    print(f"  waveforms : {result['Number of waveforms']}")
    print(f"  accepted  : {result['Accepted pulses N_acc']}")
    print(f"  total Q   : {result['Total accepted charge [C]']:.4e} C")
    print(f"  peak Q    : {result['Charge peak [C]']:.4e} C")

print("Done")

## Dark-Reference Comparisons

This table compares each test condition to the dark reference at the same voltage. It reports differences and ratios for total accepted charge, normalized accepted charge per waveform, and charge peak.


In [ ]:
# -----------------------------
# Print comparison table
# -----------------------------


print("========== TRANSPARENCY CHARGE COMPARISON ==========")
for comparison in comparison_specs:
    voltage = comparison["voltage"]
    ref_label = comparison["reference_label"]
    test_label = comparison["test_label"]
    reference = get_result(transparency_results, ref_label, voltage)
    test = get_result(transparency_results, test_label, voltage)

    if reference is None or test is None:
        continue

    total_excess = test["Total accepted charge [C]"] - reference["Total accepted charge [C]"]
    total_excess_err = np.sqrt(test["Total accepted charge error [C]"]**2 + reference["Total accepted charge error [C]"]**2)
    total_ratio = test["Total accepted charge [C]"] / reference["Total accepted charge [C]"] if reference["Total accepted charge [C]"] > 0 else np.nan
    total_ratio_err = total_ratio * np.sqrt(
        (test["Total accepted charge error [C]"] / test["Total accepted charge [C]"])**2
        + (reference["Total accepted charge error [C]"] / reference["Total accepted charge [C]"])**2
    ) if test["Total accepted charge [C]"] > 0 and reference["Total accepted charge [C]"] > 0 else np.nan

    per_waveform_excess = test["Charge per waveform [C/wf]"] - reference["Charge per waveform [C/wf]"]
    per_waveform_excess_err = np.sqrt(test["Charge per waveform error [C/wf]"]**2 + reference["Charge per waveform error [C/wf]"]**2)
    per_waveform_ratio = test["Charge per waveform [C/wf]"] / reference["Charge per waveform [C/wf]"] if reference["Charge per waveform [C/wf]"] > 0 else np.nan
    per_waveform_ratio_err = per_waveform_ratio * np.sqrt(
        (test["Charge per waveform error [C/wf]"] / test["Charge per waveform [C/wf]"])**2
        + (reference["Charge per waveform error [C/wf]"] / reference["Charge per waveform [C/wf]"])**2
    ) if test["Charge per waveform [C/wf]"] > 0 and reference["Charge per waveform [C/wf]"] > 0 else np.nan

    peak_excess = test["Charge peak [C]"] - reference["Charge peak [C]"]
    peak_excess_err = np.sqrt(test["Charge peak error [C]"]**2 + reference["Charge peak error [C]"]**2)
    peak_ratio = test["Charge peak [C]"] / reference["Charge peak [C]"] if reference["Charge peak [C]"] > 0 else np.nan
    peak_ratio_err = peak_ratio * np.sqrt(
        (test["Charge peak error [C]"] / test["Charge peak [C]"])**2
        + (reference["Charge peak error [C]"] / reference["Charge peak [C]"])**2
    ) if test["Charge peak [C]"] > 0 and reference["Charge peak [C]"] > 0 else np.nan

    print(f"\n{voltage} V: {ref_label} vs {test_label}")
    print(f"  {ref_label} total accepted charge : {reference['Total accepted charge [C]']:.4e} +/- {reference['Total accepted charge error [C]']:.4e} C")
    print(f"  {test_label} total accepted charge : {test['Total accepted charge [C]']:.4e} +/- {test['Total accepted charge error [C]']:.4e} C")
    print(f"  {test_label} - {ref_label} total charge : {total_excess:.4e} +/- {total_excess_err:.4e} C")
    print(f"  {test_label} / {ref_label} total charge : {total_ratio:.4f} +/- {total_ratio_err:.4f}")
    print(f"  {ref_label} charge per waveform : {reference['Charge per waveform [C/wf]']:.4e} +/- {reference['Charge per waveform error [C/wf]']:.4e} C/wf")
    print(f"  {test_label} charge per waveform : {test['Charge per waveform [C/wf]']:.4e} +/- {test['Charge per waveform error [C/wf]']:.4e} C/wf")
    print(f"  {test_label} - {ref_label} charge per waveform : {per_waveform_excess:.4e} +/- {per_waveform_excess_err:.4e} C/wf")
    print(f"  {test_label} / {ref_label} charge per waveform : {per_waveform_ratio:.4f} +/- {per_waveform_ratio_err:.4f}")
    print(f"  {ref_label} charge peak : {reference['Charge peak [C]']:.4e} +/- {reference['Charge peak error [C]']:.4e} C")
    print(f"  {test_label} charge peak : {test['Charge peak [C]']:.4e} +/- {test['Charge peak error [C]']:.4e} C")
    print(f"  {test_label} - {ref_label} charge peak : {peak_excess:.4e} +/- {peak_excess_err:.4e} C")
    print(f"  {test_label} / {ref_label} charge peak : {peak_ratio:.4f} +/- {peak_ratio_err:.4f}")

## Accepted-Charge Distributions

These plots overlay the accepted charge histograms for each test/reference pair. The dashed lines mark the estimated charge peaks used as a diagnostic of pulse size.


In [ ]:
# -----------------------------
# Overlay charge distributions
# -----------------------------
for comparison in comparison_specs:
    voltage = comparison["voltage"]
    ref_label = comparison["reference_label"]
    test_label = comparison["test_label"]
    reference = get_result(transparency_results, ref_label, voltage)
    test = get_result(transparency_results, test_label, voltage)

    if reference is None or test is None:
        continue

    plt.figure(figsize=(9, 5))
    if len(reference["Accepted charges"]):
        plt.hist(reference["Accepted charges"], bins=CHARGE_HIST_BINS, histtype="step", lw=2, label=ref_label)
        plt.axvline(reference["Charge peak [C]"], ls="--", lw=1.5, label=f"{ref_label} peak")
    if len(test["Accepted charges"]):
        plt.hist(test["Accepted charges"], bins=CHARGE_HIST_BINS, histtype="step", lw=2, label=test_label)
        plt.axvline(test["Charge peak [C]"], ls="--", lw=1.5, label=f"{test_label} peak")

    plt.xlabel("Pulse-region Integrated Charge Q [C]")
    plt.ylabel("Counts")
    plt.title(f"Accepted Charge Distribution, {voltage} V")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Average Accepted-Waveform Checks

These plots compare only the average baseline-subtracted filtered waveforms that passed the pulse-selection cuts. This focuses the comparison on accepted pulse shapes and avoids the extra all-waveform panel.


In [ ]:
# -----------------------------
# Average accepted waveform comparison
# -----------------------------
# These are baseline-subtracted, filtered averages using only waveforms
# that pass the pulse-selection cuts.
for comparison in comparison_specs:
    voltage = comparison["voltage"]
    ref_label = comparison["reference_label"]
    test_label = comparison["test_label"]
    reference = get_result(transparency_results, ref_label, voltage)
    test = get_result(transparency_results, test_label, voltage)

    if reference is None or test is None:
        continue

    plt.figure(figsize=(8, 4.5))

    for result, color in [(reference, "tab:blue"), (test, "tab:orange")]:
        t_acc_ns = result.get("Average time accepted [s]", result.get("Average time [s]", np.array([]))) * 1e9

        if len(result["Average waveform accepted [V]"]) and len(t_acc_ns) == len(result["Average waveform accepted [V]"]):
            plt.plot(
                t_acc_ns,
                result["Average waveform accepted [V]"] * 1e3,
                lw=2,
                color=color,
                label=f"{result['Label']} (N={result['Average waveform accepted N']})",
            )

    plt.axhline(0, ls=":", lw=1)
    plt.title(f"{voltage} V: Accepted Pulse Average")
    plt.xlabel("Time [ns]")
    plt.ylabel("Average Voltage [mV]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Test / Dark Ratio Summary

This cell condenses the same-voltage comparisons into ratio plots. Ratios above one mean the test condition has more normalized accepted charge or a larger charge peak than the dark reference.


In [ ]:
# -----------------------------
# Test / reference ratios with propagated error bars
# -----------------------------
ratio_labels = []
charge_per_waveform_ratios = []
charge_per_waveform_ratio_errors = []
charge_peak_ratios = []
charge_peak_ratio_errors = []

for comparison in comparison_specs:
    voltage = comparison["voltage"]
    ref_label = comparison["reference_label"]
    test_label = comparison["test_label"]
    reference = get_result(transparency_results, ref_label, voltage)
    test = get_result(transparency_results, test_label, voltage)

    if reference is None or test is None:
        continue

    ratio_labels.append(f"{voltage} V\n{test_label} / {ref_label}")

    cpw_ratio = test["Charge per waveform [C/wf]"] / reference["Charge per waveform [C/wf]"]
    cpw_ratio_err = cpw_ratio * np.sqrt(
        (test["Charge per waveform error [C/wf]"] / test["Charge per waveform [C/wf]"])**2
        + (reference["Charge per waveform error [C/wf]"] / reference["Charge per waveform [C/wf]"])**2
    )
    charge_per_waveform_ratios.append(cpw_ratio)
    charge_per_waveform_ratio_errors.append(cpw_ratio_err)

    peak_ratio = test["Charge peak [C]"] / reference["Charge peak [C]"]
    peak_ratio_err = peak_ratio * np.sqrt(
        (test["Charge peak error [C]"] / test["Charge peak [C]"])**2
        + (reference["Charge peak error [C]"] / reference["Charge peak [C]"])**2
    )
    charge_peak_ratios.append(peak_ratio)
    charge_peak_ratio_errors.append(peak_ratio_err)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].errorbar(ratio_labels, charge_per_waveform_ratios, yerr=charge_per_waveform_ratio_errors, fmt="o", capsize=5)
axes[0].axhline(1.0, ls="--", lw=1.5)
axes[0].set_ylabel("Test / Reference")
axes[0].set_title("Charge per Waveform Ratio")
axes[0].grid(True, alpha=0.3)

axes[1].errorbar(ratio_labels, charge_peak_ratios, yerr=charge_peak_ratio_errors, fmt="o", capsize=5)
axes[1].axhline(1.0, ls="--", lw=1.5)
axes[1].set_ylabel("Test / Reference")
axes[1].set_title("Charge Peak Ratio")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Final Normalized-Charge Plot

This is the main absolute comparison plot. It includes the original voltage series and the three appended 900 V datasets.


In [ ]:
# -----------------------------
# Final normalized-charge plot
# -----------------------------
# Charge per waveform is the total accepted charge divided by the total number
# of waveforms in each dataset.
plot_order = {
    "Dark": 0,
    "Dark 06_07": 1,
    "Without Window": 2,
    "Without Window 6_5": 3,
    "With Window": 4,
    "With Window 6_3": 5,
}
plot_colors = {
    "Dark": "tab:gray",
    "Dark 06_07": "black",
    "Without Window": "tab:green",
    "Without Window 6_5": "tab:olive",
    "With Window": "tab:blue",
    "With Window 6_3": "tab:cyan",
}
plot_results = sorted(
    [result for result in transparency_results if result["Status"] == "OK"],
    key=lambda result: (result["Voltage [V]"], plot_order.get(result["Label"], 99)),
)

labels = [f"{result['Voltage [V]']} V\n{result['Label']}" for result in plot_results]
charge_per_waveform = [result["Charge per waveform [C/wf]"] for result in plot_results]
charge_per_waveform_errors = [result["Charge per waveform error [C/wf]"] for result in plot_results]
colors = [plot_colors.get(result["Label"], "tab:purple") for result in plot_results]

plt.figure(figsize=(14, 5.5))
plt.bar(
    labels,
    charge_per_waveform,
    yerr=charge_per_waveform_errors,
    capsize=4,
    color=colors,
    edgecolor="black",
    linewidth=0.8,
)
plt.ylabel("Accepted Charge per Waveform [C/wf]")
plt.title("Normalized Total Accepted Charge by Dataset")
plt.xticks(rotation=35, ha="right")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Voltage-Ratio Plot: 800 V / 700 V

This plot compares how the normalized accepted charge changes from 700 V to 800 V within each condition. It is not a window/dark ratio; it is a voltage-scaling ratio computed separately for Dark, Without Window, and With Window.


In [ ]:
# -----------------------------
# Final ratio plot: 800 V / 700 V normalized accepted charge
# -----------------------------
ratio_dataset_labels = ["Dark", "Without Window", "With Window"]
ratio_low_voltage = 700
ratio_high_voltage = 800

ratio_values = []
ratio_errors = []
ratio_plot_labels = []

for label in ratio_dataset_labels:
    low = get_result(label, ratio_low_voltage)
    high = get_result(label, ratio_high_voltage)

    if low is None or high is None:
        print(f"Missing {label} {ratio_low_voltage} V or {ratio_high_voltage} V result")
        continue

    q_low = low["Charge per waveform [C/wf]"]
    q_high = high["Charge per waveform [C/wf]"]
    err_low = low["Charge per waveform error [C/wf]"]
    err_high = high["Charge per waveform error [C/wf]"]

    ratio = q_high / q_low if q_low > 0 else np.nan
    ratio_err = (
        ratio * np.sqrt((err_high / q_high)**2 + (err_low / q_low)**2)
        if q_high > 0 and q_low > 0 and np.isfinite(err_high) and np.isfinite(err_low)
        else np.nan
    )

    ratio_plot_labels.append(label)
    ratio_values.append(ratio)
    ratio_errors.append(ratio_err)

    print(f"{label}: ({ratio_high_voltage} V / {ratio_low_voltage} V) = {ratio:.4f} +/- {ratio_err:.4f}")

colors = [
    "tab:gray" if label == "Dark"
    else "tab:green" if label == "Without Window"
    else "tab:blue"
    for label in ratio_plot_labels
]

plt.figure(figsize=(7, 5))
plt.bar(
    ratio_plot_labels,
    ratio_values,
    yerr=ratio_errors,
    capsize=5,
    color=colors,
    edgecolor="black",
    linewidth=0.8,
)
plt.axhline(1.0, ls="--", lw=1.5, color="black", alpha=0.7)
plt.ylabel(f"Normalized Accepted Charge Ratio ({ratio_high_voltage} V / {ratio_low_voltage} V)")
plt.title(f"{ratio_high_voltage} V / {ratio_low_voltage} V Charge Ratio")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Voltage-Ratio Plot: 900 V / 800 V

This plot repeats the voltage-scaling comparison for 900 V divided by 800 V.


In [ ]:
# -----------------------------
# Final ratio plot: 900 V / 800 V normalized accepted charge
# -----------------------------
ratio_dataset_labels = ["Dark", "Without Window", "With Window"]
ratio_low_voltage = 800
ratio_high_voltage = 900

ratio_values = []
ratio_errors = []
ratio_plot_labels = []

for label in ratio_dataset_labels:
    low = get_result(label, ratio_low_voltage)
    high = get_result(label, ratio_high_voltage)

    if low is None or high is None:
        print(f"Missing {label} {ratio_low_voltage} V or {ratio_high_voltage} V result")
        continue

    q_low = low["Charge per waveform [C/wf]"]
    q_high = high["Charge per waveform [C/wf]"]
    err_low = low["Charge per waveform error [C/wf]"]
    err_high = high["Charge per waveform error [C/wf]"]

    ratio = q_high / q_low if q_low > 0 else np.nan
    ratio_err = (
        ratio * np.sqrt((err_high / q_high)**2 + (err_low / q_low)**2)
        if q_high > 0 and q_low > 0 and np.isfinite(err_high) and np.isfinite(err_low)
        else np.nan
    )

    ratio_plot_labels.append(label)
    ratio_values.append(ratio)
    ratio_errors.append(ratio_err)

    print(f"{label}: ({ratio_high_voltage} V / {ratio_low_voltage} V) = {ratio:.4f} +/- {ratio_err:.4f}")

colors = [
    "tab:gray" if label == "Dark"
    else "tab:green" if label == "Without Window"
    else "tab:blue"
    for label in ratio_plot_labels
]

plt.figure(figsize=(7, 5))
plt.bar(
    ratio_plot_labels,
    ratio_values,
    yerr=ratio_errors,
    capsize=5,
    color=colors,
    edgecolor="black",
    linewidth=0.8,
)
plt.axhline(1.0, ls="--", lw=1.5, color="black", alpha=0.7)
plt.ylabel(f"Normalized Accepted Charge Ratio ({ratio_high_voltage} V / {ratio_low_voltage} V)")
plt.title(f"{ratio_high_voltage} V / {ratio_low_voltage} V Charge Ratio")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 900 V New-Run Comparison Plots

These plots focus on the three 900 V run-to-run comparisons requested: with-window normalized datasets, without-window normalized datasets, and dark datasets. The cumulative plot shows all three pairs together.


In [ ]:
# -----------------------------
# 900 V focused comparison plots
# -----------------------------
focused_900_groups = [
    {
        "title": "With Window Normalized Charge, 900 V",
        "labels": ["With Window", "With Window 6_3"],
        "colors": ["tab:blue", "tab:cyan"],
    },
    {
        "title": "Without Window Normalized Charge, 900 V",
        "labels": ["Without Window", "Without Window 6_5"],
        "colors": ["tab:green", "tab:olive"],
    },
    {
        "title": "Dark Normalized Charge, 900 V",
        "labels": ["Dark", "Dark 06_07"],
        "colors": ["tab:gray", "black"],
    },
]


configure_transparency(**globals())


plt.ylabel("Accepted Charge per Waveform [C/wf]")
plt.title(group["title"])
plt.yscale("log")
plt.grid(True, axis="y", alpha=0.3, which="both")
plt.tight_layout()
plt.show()


for group in focused_900_groups:
    print("\n" + group["title"])
    plot_focused_group(group)

# Cumulative grouped comparison: old 900 V run vs new 900 V run for each condition.
conditions = ["Dark", "Without Window", "With Window"]
old_labels = ["Dark", "Without Window", "With Window"]
new_labels = ["Dark 06_07", "Without Window 6_5", "With Window 6_3"]
old_values, old_errors, new_values, new_errors = [], [], [], []

for old_label, new_label in zip(old_labels, new_labels):
    value, error = result_value_and_error(old_label, 900)
    old_values.append(value)
    old_errors.append(error)

    value, error = result_value_and_error(new_label, 900)
    new_values.append(value)
    new_errors.append(error)

x = np.arange(len(conditions))
width = 0.38

plt.figure(figsize=(9, 5))
plt.bar(
    x - width / 2,
    old_values,
    width,
    yerr=old_errors,
    capsize=4,
    label="Original 900 V dataset",
    color="lightsteelblue",
    edgecolor="black",
    linewidth=0.8,
)
plt.bar(
    x + width / 2,
    new_values,
    width,
    yerr=new_errors,
    capsize=4,
    label="New 900 V dataset",
    color="salmon",
    edgecolor="black",
    linewidth=0.8,
)
plt.xticks(x, conditions)
plt.ylabel("Accepted Charge per Waveform [C/wf]")
plt.title("Cumulative 900 V Normalized-Charge Comparison")
plt.yscale("log")
plt.legend()
plt.grid(True, axis="y", alpha=0.3, which="both")
plt.tight_layout()
plt.show()